# Financial Compliance RAG — End-to-End Workflow

> Orchestrates `src/` modules step-by-step. See [docs/PROJECT_SPEC.md](../docs/PROJECT_SPEC.md).

## Step 1: Problem Definition — Compliance Document Search Pain Point

In [ ]:
from src.config import get_settings

settings = get_settings()
print("PDF directory:", settings.raw_pdf_path)
print("Chunk size:", settings.chunk_size, "| overlap:", settings.chunk_overlap)

## Step 2: Data Collection — Download Regulatory & Filing PDFs

In [ ]:
# Run once to fetch PDFs (or place files manually in data/raw_pdfs/)
# !python ../scripts/download_docs.py

from pathlib import Path
from src.config import get_settings

pdf_dir = get_settings().raw_pdf_path
pdfs = sorted(pdf_dir.glob("*.pdf"))
print(f"Found {len(pdfs)} PDF(s):")
for p in pdfs:
    print(" -", p.name)

## Step 3: Document Ingestion — PDF Text Extraction (PyMuPDF)

In [ ]:
from src.ingest_docs import ingest_directory

chunks = ingest_directory()
print(f"Ingested {len(chunks)} chunks")
if chunks:
    sample = chunks[0]
    print(f"Sample: {sample.source} p.{sample.page} — {sample.text[:120]}...")

## Step 4: Chunking — Paragraph-Aware Text Splitting

## Step 5: Embeddings — Sentence-Transformers (MiniLM)

In [ ]:
from src.embeddings import EMBEDDING_DIMENSION, embed_query

vector = embed_query("What are the KYC requirements for banks?")
print(f"Embedding dimension: {len(vector)} (expected {EMBEDDING_DIMENSION})")

## Step 6: Vector Store — ChromaDB Persistence

In [ ]:
from src.vectorstore import build_vector_index, get_collection_count

stored = build_vector_index()
print(f"Stored {stored} chunks in ChromaDB")
print(f"Collection count: {get_collection_count()}")

## Step 7: Retrieval — Top-k Semantic Search

In [ ]:
from src.retriever import retrieve

results = retrieve("What KYC documents are required for individual customers?", top_k=5)
print(f"Retrieved {len(results)} chunks")
for r in results[:3]:
    print(f"- {r.source} p.{r.page} score={r.score:.4f}")
    print(f"  {r.text[:120]}...")

## Step 8: Generation — OpenRouter LLM with Citations

In [ ]:
from src.generator import generate_answer

question = "What KYC documents are required for individual customers?"
contexts = retrieve(question, top_k=5)
response = generate_answer(question, contexts)

print(response.answer)
print("\nCitations:")
for c in response.citations:
    print(f"- {c.source} p.{c.page}")

## Step 9: Evaluation — Sample Compliance Questions

## Step 10: Deployment — Streamlit Chat UI + Docker